In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 23


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.562464617192745
Epoch 2/100, Loss: 3.0825773775577545
Epoch 3/100, Loss: 3.2747222185134888
Epoch 4/100, Loss: 3.1173048615455627
Epoch 5/100, Loss: 3.0672856718301773
Epoch 6/100, Loss: 2.976446993649006
Epoch 7/100, Loss: 3.1036886982619762
Epoch 8/100, Loss: 2.830881841480732
Epoch 9/100, Loss: 3.196855530142784
Epoch 10/100, Loss: 3.6492850109934807
Epoch 11/100, Loss: 2.617651127278805
Epoch 12/100, Loss: 2.7386743426322937
Epoch 13/100, Loss: 2.71440452337265
Epoch 14/100, Loss: 3.014733076095581
Epoch 15/100, Loss: 2.97062174230814
Epoch 16/100, Loss: 2.9356360882520676
Epoch 17/100, Loss: 2.9588484540581703


Epoch 18/100, Loss: 2.957296662032604
Epoch 19/100, Loss: 3.1859570741653442
Epoch 20/100, Loss: 3.2414165288209915
Epoch 21/100, Loss: 2.7384480088949203
Epoch 22/100, Loss: 3.379339098930359
Epoch 23/100, Loss: 3.450148805975914
Epoch 24/100, Loss: 3.3471796587109566
Epoch 25/100, Loss: 3.80577552318573
Epoch 26/100, Loss: 3.7541777342557907
Epoch 27/100, Loss: 2.845496214926243
Epoch 28/100, Loss: 3.3125612139701843


Epoch 29/100, Loss: 3.406961739063263
Epoch 30/100, Loss: 3.3818040788173676
Epoch 31/100, Loss: 2.740839086472988
Epoch 32/100, Loss: 3.3032310605049133
Epoch 33/100, Loss: 2.8672487288713455
Epoch 34/100, Loss: 3.101403683423996
Epoch 35/100, Loss: 3.615751586854458
Epoch 36/100, Loss: 2.996363252401352
Epoch 37/100, Loss: 3.07197854667902
Epoch 38/100, Loss: 3.0923559442162514
Epoch 39/100, Loss: 3.100825823843479


Epoch 40/100, Loss: 3.103971891105175
Epoch 41/100, Loss: 3.4001335874199867
Epoch 42/100, Loss: 2.9466065913438797
Epoch 43/100, Loss: 3.2437249198555946
Epoch 44/100, Loss: 3.5384989827871323
Epoch 45/100, Loss: 3.4338733106851578
Epoch 46/100, Loss: 3.3928795903921127
Epoch 47/100, Loss: 3.6676183715462685
Epoch 48/100, Loss: 3.1889449805021286
Epoch 49/100, Loss: 3.337876372039318
Epoch 50/100, Loss: 3.5777102187275887


Epoch 51/100, Loss: 3.5681047514081
Epoch 52/100, Loss: 3.122712768614292
Epoch 53/100, Loss: 3.3732605054974556
Epoch 54/100, Loss: 3.2642814740538597
Epoch 55/100, Loss: 2.9634354896843433
Epoch 56/100, Loss: 3.4347823560237885
Epoch 57/100, Loss: 3.1214505806565285
Epoch 58/100, Loss: 2.784368373453617
Epoch 59/100, Loss: 3.2281971126794815
Epoch 60/100, Loss: 3.2450340017676353
Epoch 61/100, Loss: 3.362050585448742


...

Epoch 13/100, Loss: 2.8621731400489807
Epoch 14/100, Loss: 2.7932237088680267
Epoch 15/100, Loss: 3.0441326797008514
Epoch 16/100, Loss: 2.9668832421302795
Epoch 17/100, Loss: 2.7273417487740517
Epoch 18/100, Loss: 3.174343705177307
Epoch 19/100, Loss: 2.7376217171549797
Epoch 20/100, Loss: 3.147294759750366
Epoch 21/100, Loss: 2.7477276250720024
Epoch 22/100, Loss: 2.8172657415270805
Epoch 23/100, Loss: 2.9956475347280502
Epoch 24/100, Loss: 2.8897750601172447
Epoch 25/100, Loss: 2.9239947348833084
Epoch 26/100, Loss: 3.059881702065468
Epoch 27/100, Loss: 2.946924425661564
Epoch 28/100, Loss: 2.8369186967611313
Epoch 29/100, Loss: 2.793608747422695
Epoch 30/100, Loss: 2.8078459054231644


Epoch 31/100, Loss: 3.0241686552762985
Epoch 32/100, Loss: 2.7010071128606796
Epoch 33/100, Loss: 2.7096399664878845
Epoch 34/100, Loss: 2.7928435802459717
Epoch 35/100, Loss: 2.7011498510837555
Epoch 36/100, Loss: 2.8809263110160828
Epoch 37/100, Loss: 2.914757713675499
Epoch 38/100, Loss: 2.8967480808496475
Epoch 39/100, Loss: 3.0961355417966843
Epoch 40/100, Loss: 2.8759089335799217
Epoch 41/100, Loss: 2.7632723599672318
Epoch 42/100, Loss: 2.7023247107863426
Epoch 43/100, Loss: 2.9091630429029465
Epoch 44/100, Loss: 2.8952011168003082
Epoch 45/100, Loss: 2.9183623045682907
Epoch 46/100, Loss: 2.9358411729335785
Epoch 47/100, Loss: 3.039269581437111


Epoch 48/100, Loss: 2.6273336187005043
Epoch 49/100, Loss: 3.007330536842346
Epoch 50/100, Loss: 3.080989897251129
Epoch 51/100, Loss: 3.0794434994459152
Epoch 52/100, Loss: 2.9119133353233337
Epoch 53/100, Loss: 2.9560037180781364
Epoch 54/100, Loss: 2.868828073143959
Epoch 55/100, Loss: 2.846837282180786
Epoch 56/100, Loss: 2.9094676226377487
Epoch 57/100, Loss: 2.747522532939911
Epoch 58/100, Loss: 2.7409989535808563
Epoch 59/100, Loss: 2.893196776509285
Epoch 60/100, Loss: 3.024896964430809
Epoch 61/100, Loss: 3.1854988262057304
Epoch 62/100, Loss: 2.8326840102672577
Epoch 63/100, Loss: 3.0003815293312073
Epoch 64/100, Loss: 2.8550995737314224
Epoch 65/100, Loss: 2.971591666340828


Epoch 66/100, Loss: 2.9902091175317764
Epoch 67/100, Loss: 2.9067598283290863
Epoch 68/100, Loss: 2.972569413483143
Epoch 69/100, Loss: 3.034905605018139
Epoch 70/100, Loss: 2.8536883518099785
Epoch 71/100, Loss: 2.995185635983944
Epoch 72/100, Loss: 2.8841214329004288
Epoch 73/100, Loss: 3.083922356367111
Epoch 74/100, Loss: 2.8796823918819427
Epoch 75/100, Loss: 2.8982683271169662
Epoch 76/100, Loss: 2.9200063049793243
Epoch 77/100, Loss: 2.8342673927545547
Epoch 78/100, Loss: 2.9362602531909943
Epoch 79/100, Loss: 2.827209174633026
Epoch 80/100, Loss: 3.0299170464277267
Epoch 81/100, Loss: 2.727018468081951
Epoch 82/100, Loss: 2.9811390936374664
Epoch 83/100, Loss: 2.7692444398999214


Epoch 84/100, Loss: 2.896260656416416
Epoch 85/100, Loss: 2.878194972872734
Epoch 86/100, Loss: 2.9395750761032104
Epoch 87/100, Loss: 2.8350622802972794
Epoch 88/100, Loss: 2.9460655599832535
Epoch 89/100, Loss: 3.072058320045471
Epoch 90/100, Loss: 2.7935170233249664
Epoch 91/100, Loss: 2.9992001056671143
Epoch 92/100, Loss: 2.9622403010725975
Epoch 93/100, Loss: 2.937581218779087
Epoch 94/100, Loss: 3.0307158157229424
Epoch 95/100, Loss: 3.1520219296216965
Epoch 96/100, Loss: 2.9948222041130066
Epoch 97/100, Loss: 2.7886072024703026
Epoch 98/100, Loss: 2.863723561167717
Epoch 99/100, Loss: 3.4126179441809654
Epoch 100/100, Loss: 2.9391247630119324
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.4936
